# 05 — Lagrange multipliers and $A\mathbf{x}=\mathbf{0}$

**Needed for the assignment:** $\bigstar$ YES. **Study:** 1.5 h, **Practice:** 1 h.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Idea

**Lagrange multiplier:** to minimise $f(\mathbf{x})$ *subject to* $g(\mathbf{x})=c$, add a new unknown $\lambda$ and minimise $J=f-\lambda(g-c)$ freely. At the optimum the gradients of $f$ and $g$ are parallel.\
**Homogeneous system $A\mathbf{x}=\mathbf{0}$:** $\mathbf{x}=\mathbf{0}$ always works, but it is useless. So we ask for $\min\|A\mathbf{x}\|^2$ *subject to* $\|\mathbf{x}\|=1$. Lagrange then turns this into an eigenvalue problem.

## Key formulas

## Lagrange Multipliers

**General constrained optimization**

To minimize $f(\mathbf{x})$ subject to $g(\mathbf{x})=c$, define the Lagrangian:

$$
J=f(\mathbf{x})-\lambda(g(\mathbf{x})-c)
$$

Set the partial derivatives to zero:

$$
\frac{\partial J}{\partial\mathbf{x}}=\mathbf{0}
\quad\Longrightarrow\quad
\nabla f=\lambda\nabla g
$$

$$
\frac{\partial J}{\partial\lambda}=0
\quad\Longrightarrow\quad
g(\mathbf{x})=c
$$

## Application to Finding the Smallest Eigenvalue

Minimize the quadratic form subject to a unit-length constraint:

$$
J=\mathbf{x}^TA^TA\mathbf{x}
-\lambda(\mathbf{x}^T\mathbf{x}-1)
$$

Differentiate with respect to $\mathbf{x}$ and set equal to zero:

$$
2A^TA\mathbf{x}-2\lambda\mathbf{x}=\mathbf{0}
$$

Rearrange:

$$
\boxed{A^TA\mathbf{x}=\lambda\mathbf{x}}
$$

This is the eigenvalue equation for $A^TA$.

**Why choose the smallest eigenvalue?**

The cost function is:

$$
\begin{aligned}
\text{cost}
&=\mathbf{x}^TA^TA\mathbf{x}\\
&=\lambda\mathbf{x}^T\mathbf{x}\\
&=\lambda
\end{aligned}
$$

The last equality follows from the constraint $\mathbf{x}^T\mathbf{x}=1$.

Therefore, to minimize the cost, choose:

$$
\boxed{
\mathbf{x}=\text{eigenvector of }A^TA
\text{ corresponding to its smallest eigenvalue}
}
$$

## Scale Ambiguity

The solution is defined only up to scale. We can rescale it afterwards.

For example, if the solution is represented by a homography matrix $H$, we can normalize it by its last element:

$$
H_{\text{normalized}}=\frac{H}{H_{3,3}}
$$

In NumPy:

```python
H = H / H[2, 2]
```

This normalization assumes $H[2,2]\neq0$.

# Worked example 1 (on paper): Lagrange in 2D

Minimise $f=x^2+y^2$ subject to $x+y=2$ (the point of the line closest to the origin). $$J=x^2+y^2-\lambda(x+y-2)$$ $$\frac{\partial J}{\partial x}=2x-\lambda=0,\qquad \frac{\partial J}{\partial y}=2y-\lambda=0,\qquad \frac{\partial J}{\partial\lambda}=-(x+y-2)=0$$ From the first two equations $x=y=\lambda/2$. Put this in the constraint: $\lambda/2+\lambda/2=2\Rightarrow\lambda=2$, so $x=y=1$ and $f=2$.

# Worked example 2 (on paper): $A\mathbf{x}=\mathbf{0}$

$A=\begin{bmatrix}1&1\\0&1\\1&0\end{bmatrix}$. Three equations $x_1+x_2=0,\ x_2=0,\ x_1=0$ have only the trivial exact solution, so we minimise.

**1.** $A^TA=\begin{bmatrix}2&1\\1&2\end{bmatrix}$, with eigenvalues $1$ and $3$ (topic 03).

**2.** The smallest eigenvalue is $\lambda=1$, with eigenvector $\mathbf{x}^*=\tfrac1{\sqrt2}(1,-1)$.

**3. Check the cost.** $A\mathbf{x}^*=\tfrac1{\sqrt2}(0,\,-1,\,1)$, so $\|A\mathbf{x}^*\|^2=\tfrac12(0+1+1)=1=\lambda$ ✓. Any other unit vector gives more; e.g. $(1,0)$: $A\mathbf{x}=(1,0,1)$, cost $2$.

# The same in Python / NumPy

In [2]:
import numpy as np

# --- 1) Lagrange in 2D: minimise x^2 + y^2 subject to x + y = 2 ---
# from the derivation: x = y = lambda/2, and x + y = 2  ->  lambda = 2
lam = 2.0
x = y = lam / 2
print("optimum (x, y) =", (x, y), " f =", x**2 + y**2)

# brute-force check on the constraint line
t = np.linspace(-3, 5, 80001)          # points (t, 2 - t)
f = t**2 + (2 - t)**2
print("brute force min at x =", round(t[np.argmin(f)], 4), " f =", round(f.min(), 4))

# --- 2) homogeneous system A x = 0 with ||x|| = 1 ---
A = np.array([[1.0, 1.0],
              [0.0, 1.0],
              [1.0, 0.0]])
M = A.T @ A
val, vec = np.linalg.eig(M)
val, vec = val.real, vec.real      # eig may return complex dtype
x_opt = vec[:, np.argmin(val)]
print("A^T A =\n", M)
print("eigenvalues:", val)
print("x* =", np.round(x_opt, 4), " ||x*|| =", np.linalg.norm(x_opt))
print("A x* =", np.round(A @ x_opt, 4), " cost ||A x*||^2 =", round(np.sum((A @ x_opt)**2), 6))

# any other unit vector costs more
for angle in [0, 30, 60, 90, 135]:
    u = np.array([np.cos(np.radians(angle)), np.sin(np.radians(angle))])
    print(f"angle {angle:3d}:  ||A u||^2 = {np.sum((A @ u)**2):.4f}")

# same answer from SVD: last row of Vt
_, _, Vt = np.linalg.svd(A)
print("SVD last row of Vt:", np.round(Vt[-1], 4))

optimum (x, y) = (1.0, 1.0)  f = 2.0
brute force min at x = 1.0  f = 2.0
A^T A =
 [[2. 1.]
 [1. 2.]]
eigenvalues: [3. 1.]
x* = [-0.7071  0.7071]  ||x*|| = 0.9999999999999999
A x* = [ 0.      0.7071 -0.7071]  cost ||A x*||^2 = 1.0
angle   0:  ||A u||^2 = 2.0000
angle  30:  ||A u||^2 = 2.8660
angle  60:  ||A u||^2 = 2.8660
angle  90:  ||A u||^2 = 2.0000
angle 135:  ||A u||^2 = 1.0000
SVD last row of Vt: [ 0.7071 -0.7071]


## Where this is used in the assignment

Every "fit" in the solution is this recipe: one row per point, then $A^TA$, then the smallest eigenvector. Plane refinement uses the rows $[x-\bar x,\ y-\bar y,\ z-\bar z]$ (topic 06), and the sphere fit uses $[x^2+y^2+z^2,\ x,\ y,\ z,\ 1]$ (topic 07).

## Course / lab link

Lab 6 ("Solution for 1.b") and the Practice notebook derive exactly this for the homography $\mathbf{h}$ (9 unknowns) and the projection matrix $\mathbf{p}$ (12 unknowns). Lagrange-1.pdf, Section 2.

## Practice

Write your solution in the empty cell under each exercise. The **expected result** tells you what you should get, not how to get it. Grey *provided data* cells are already filled in; run them first.

### Exercise 1

Minimise $f=x^2+2y^2$ subject to $x+y=3$ with a Lagrange multiplier. Check it by brute force in NumPy.

> **Expected result:** $x=2,\ y=1,\ \lambda=4,\ f=6$.

In [3]:
# Exercise 1: your solution


### Exercise 2

$A=\begin{bmatrix}1&2\\2&4.1\\3&5.9\end{bmatrix}$: compute $A^TA$ by hand, then find $\mathbf x^*$ with `eig` and with `svd`. What is the cost?

> **Expected result:** $A^TA=\begin{bmatrix}14&27.9\\27.9&55.62\end{bmatrix}$, $\mathbf x^*\approx\pm(-0.8938,\ 0.4484)$, cost $=\lambda_{\min}\approx0.00388$.

In [4]:
# Exercise 2: your solution


### Exercise 3

Why do we need the constraint $\|\mathbf x\|=1$? What would the minimum be without it?

> **Expected result:** Without it, $\mathbf x=\mathbf 0$ gives cost 0, which is a useless trivial solution.

In [5]:
# Exercise 3: your solution


### Exercise 4

Show numerically that the eigenvector of the **largest** eigenvalue maximises $\|A\mathbf x\|^2$ for the matrix of exercise 2.

> **Expected result:** Largest eigenvalue $\approx69.616$, eigenvector $\approx\pm(0.4484,0.8938)$, and $\|A\mathbf x\|^2\approx69.616$; no other unit vector gives more.

In [6]:
# Exercise 4: your solution
